<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/healthstack-logo-dark.png" alt="HealthStack logo" width="320" />
</div>

<p align="center"><strong>AFROTECH 2026</strong></p>
<p align="center"><font size="6"><strong>Building Fairness Into the Data Supply Chain</strong></font></p>
<p align="center"><strong>Instructor: Waverly Rose Brim</strong></p>

---

<p align="center"><font size="6"><strong>How big is this really?</strong></font></p>

<div align="center">
  <img src="https://datasupplychain.waverlyrose.io/how-big.jpg" alt="Many hands of different skin tones stacked together in a circle." width="640" />
  <p><em>Thirteen patients in a room. The design is built to speak for a country.</em></p>
</div>

---

## Start here
This notebook answers three questions:
1. How many Black adults in this cohort cross into CKD under the 2021 refit?
2. How many people could that represent across the US population?
3. When is a national estimate reliable enough to report?

**Workflow:** predict -> run -> inspect -> interpret.

## Colab setup
Skip this cell if you are running locally.

In [ ]:
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps", "--no-cache-dir",
        "data-supply-chain-workshop==0.1.2",
    ])
else:
    import data_supply_chain

## Cold start
Same cohort, same split, whichever notebook you opened first. Earlier
notebooks do not need to be open.

In [ ]:
from dataclasses import replace

from data_supply_chain.bootstrap import cold_start
from data_supply_chain.egfr import add_egfr_columns

S = cold_start()
S = replace(S, df=add_egfr_columns(S.df))
print(f"{len(S.df):,} adults · source={S.source} · real={S.is_real}")

The medication counts quoted below come from step 6. They are recomputed
here so this notebook stands on its own.

In [ ]:
from data_supply_chain.decisions import medication_masking_table

dm = S.df[S.df.diabetic].copy()
mask = medication_masking_table(dm)

A result can look small when you only count the people directly in front of you.

In this cohort, the first row of the next table is the observed Black adults
who cross into CKD when the 2021 race-free refit replaces the 2009 equation.

But NHANES is not an ordinary convenience sample. It is a complex probability
sample designed to represent the US population. Each participant carries a
survey weight reflecting how many people they represent nationally, while
strata and primary sampling units preserve the structure of the survey design.

So the question is not only **how many people did we observe?** It is also
**how many people could this represent across the country?**

In [ ]:
from data_supply_chain.survey import national_projection_table, render_projection_table

if not S.is_real:
    proj = None
    print("SYNTHETIC MODE — national projection disabled by design.")
    print("Nothing here would be true of the United States.")
else:
    proj = national_projection_table(S)
    print(render_projection_table(proj).to_string(index=False))

    _h = proj.iloc[0]
    print(f"\nHEADLINE (the reliable row):\n~{_h.national_count:,.0f} "
          f"(95% CI {_h.ci_low:,.0f}-{_h.ci_high:,.0f}) Black adults nationally (US population, 1999–2004) "
          "would sit on the healthier-looking side of the CKD line under the 2009 "
          "equation rather than the 2021 refit.")
    print(f"Medication rows: report the cohort counts "
          f"({mask['black_masked_n']} patients, "
          f"{mask['black_cross_30']} into contraindication), "
          "never a precise national number.")

The CKD row is a **reportable national estimate**. The metformin row is
still a real finding in this cohort — report the observed counts from
step 6, not a national number. The limit is the **scale of the claim**,
not the finding.

---

## Maintainer cell (safe to skip in workshop)
This writes the findings the website reads, so the numbers on the page stay
synced to this notebook's output.

In [ ]:
from data_supply_chain.findings import Findings
from data_supply_chain.paths import findings_path

f = Findings.open()
if proj is not None:
    _row = proj.iloc[0]
    f.record("act2.projection.ckd.n", int(_row.unweighted_n))
    f.record("act2.projection.ckd.count", int(round(_row.national_count)))
    f.record("act2.projection.ckd.ci_low", int(round(_row.ci_low)))
    f.record("act2.projection.ckd.ci_high", int(round(_row.ci_high)))
    f.record("act2.projection.ckd.pct", round(float(_row.weighted_pct) * 100, 2))
    f.record("act2.projection.masked.n", int(proj.iloc[1].unweighted_n))
    f.record("act2.projection.masked.count", int(round(proj.iloc[1].national_count)))
    f.record("act2.projection.masked.reliable", bool(proj.iloc[1].reliable))

if findings_path() is None:
    print("Outside a repo checkout - findings.json not written (expected in Colab).")
else:
    f.write(state=S)
    print("wrote findings.json")